# Tic-tac-toe world model experiments

This notebook is a clean scratchpad. Heavy experiments should run through `make` so they stay reproducible from the terminal.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

import numpy as np
import torch

from eval_world_model import load_model, make_targets
from world_model import encode

In [ ]:
data = np.load("transitions_exhaustive.npz")
model = load_model(Path("wm.pt"))

x = encode(data["states"], data["actions"])
target_boards, target_rewards, target_dones = make_targets(data)

with torch.no_grad():
    board_logits, reward_logits, done_logits = model(x)

predicted_boards = board_logits.argmax(dim=2)
predicted_rewards = reward_logits.argmax(dim=1)
predicted_dones = torch.sigmoid(done_logits) > 0.5

board_exact = (predicted_boards == target_boards).all(dim=1).float().mean()
reward_acc = (predicted_rewards == target_rewards).float().mean()
done_acc = (predicted_dones == target_dones.bool()).float().mean()

print(f"board exact-match: {board_exact:.1%}")
print(f"reward accuracy: {reward_acc:.1%}")
print(f"done accuracy: {done_acc:.1%}")

Use the Makefile for repeatable experiment runs:

In [ ]:
!make help